In [1]:
import importlib
import os
import subprocess
import sys
from pathlib import Path

# Amazon Bedrock access (via LiteLLM). Credentials come from the standard AWS
# chain: env vars, shared config/credentials file, or an assumed role.
# Set these in your shell (do NOT hardcode secrets in the notebook):
#   export AWS_REGION=us-east-1
#   export AWS_ACCESS_KEY_ID=...   export AWS_SECRET_ACCESS_KEY=...
#   export AWS_SESSION_TOKEN=...   # if using temporary credentials
# or use a profile: export AWS_PROFILE=my-bedrock-profile
os.environ.setdefault("AWS_REGION", os.getenv("AWS_REGION", "us-east-1"))


def find_repo_root(start: Path) -> Path:
    """Find repo root that contains lllm/__init__.py."""
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / "lllm" / "__init__.py").exists():
            return p
    raise RuntimeError("Could not locate repo root containing lllm/__init__.py")


repo_root = find_repo_root(Path.cwd())
repo_parent = repo_root.parent.resolve()

# Remove wrong higher-level entries that trigger namespace import.
cleaned = []
for p in sys.path:
    try:
        if Path(p).resolve() == repo_parent:
            continue
    except Exception:
        pass
    cleaned.append(p)
sys.path = cleaned

# Ensure correct repo root is first in import path.
repo_root_str = str(repo_root)
if repo_root_str in sys.path:
    sys.path.remove(repo_root_str)
sys.path.insert(0, repo_root_str)

# If a wrong namespace module was imported earlier, clear it.
for mod in list(sys.modules):
    if mod == "lllm" or mod.startswith("lllm."):
        del sys.modules[mod]


def ensure_dep(import_name: str, pip_spec: str) -> None:
    """Install dependency into current kernel env if missing."""
    try:
        importlib.import_module(import_name)
    except Exception:
        print(f"Installing missing dependency: {pip_spec}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_spec])


# Required by this repo's lllm package.
ensure_dep("pydantic", "pydantic>=2")
ensure_dep("yaml", "pyyaml")
ensure_dep("litellm", "litellm")
# Required by the statistical forecasting / anomaly detection module.
ensure_dep("numpy", "numpy")
ensure_dep("pandas", "pandas")
ensure_dep("statsmodels", "statsmodels")

print("Python executable:", sys.executable)
print("repo_root:", repo_root)
print("sys.path[0]:", sys.path[0])
print("AWS_REGION:", os.environ.get("AWS_REGION"))
# Check the full AWS credential chain (env vars, shared ~/.aws/credentials
# default profile, assumed role), not just environment variables.
try:
    import boto3
    _creds_present = boto3.Session().get_credentials() is not None
except Exception:
    _creds_present = any(k in os.environ for k in ("AWS_ACCESS_KEY_ID", "AWS_PROFILE"))
print("AWS creds present:", _creds_present)

Python executable: /Users/zequn/Library/CloudStorage/OneDrive-amazon.com/lllm/Packages/time_series_analytics/.venv/bin/python
repo_root: /Users/zequn/Library/CloudStorage/OneDrive-amazon.com/lllm
sys.path[0]: /Users/zequn/Library/CloudStorage/OneDrive-amazon.com/lllm
AWS_REGION: us-east-1
AWS creds present: True


In [2]:
import lllm, sys; print(lllm, getattr(lllm, "__file__", None)); print(sys.path[:3])

<module 'lllm' from '/Users/zequn/Library/CloudStorage/OneDrive-amazon.com/lllm/lllm/__init__.py'> /Users/zequn/Library/CloudStorage/OneDrive-amazon.com/lllm/lllm/__init__.py
['/Users/zequn/Library/CloudStorage/OneDrive-amazon.com/lllm', '/opt/homebrew/Cellar/python@3.14/3.14.6/Frameworks/Python.framework/Versions/3.14/lib/python314.zip', '/opt/homebrew/Cellar/python@3.14/3.14.6/Frameworks/Python.framework/Versions/3.14/lib/python3.14']


In [3]:
from lllm.core.runtime import load_runtime
from lllm.core.config import resolve_config
from lllm.core.tactic import build_tactic
from lllm.invokers.litellm import LiteLLMInvoker

# Hotfix for providers (e.g., some Bedrock models) that return response_cost=None.
_original_build_usage = LiteLLMInvoker._build_usage


def _build_usage_safe(self, usage_dict, response_obj, model):
    usage_dict = usage_dict or {}
    usage = _original_build_usage(self, usage_dict, response_obj, model)

    # Ensure all numeric cost fields are floats, never None.
    for key in (
        "response_cost",
        "prompt_cost",
        "completion_cost",
        "input_cost_per_token",
        "output_cost_per_token",
        "cache_read_input_token_cost",
    ):
        val = usage.get(key)
        usage[key] = float(val) if val is not None else 0.0
    return usage


LiteLLMInvoker._build_usage = _build_usage_safe

pkg_toml = repo_root / "Packages" / "time_series_analytics" / "lllm.toml"
runtime = load_runtime(
    toml_path=str(pkg_toml),
    name="ts_notebook_runtime",
    discover_shared_packages=False,
)

# Mixed-model setup (the numeric forecast is computed by a statistical model;
# the LLMs only interpret it). Haiku for profiling/synthesis, Sonnet for the
# forecast-interpretation step. Models are cross-region inference profiles.
cfg = resolve_config("time_series_analytics:balanced", runtime=runtime)

tactic = build_tactic(cfg, runtime=runtime)
print("Tactic ready:", tactic.name)

Tactic ready: time_series_analysis


In [4]:
import csv
from dataclasses import dataclass

from tactics.time_series_analysis import TimeSeriesTask


def read_csv_as_text(path: Path, max_rows: int = 300) -> str:
    with path.open("r", encoding="utf-8", newline="") as fh:
        reader = csv.DictReader(fh)
        if not reader.fieldnames:
            raise ValueError(f"CSV has no header: {path}")

        rows = []
        for i, row in enumerate(reader):
            rows.append(row)
            if i + 1 >= max_rows:
                break

    header = ",".join(reader.fieldnames)
    body = [",".join(str(r.get(col, "")) for col in reader.fieldnames) for r in rows]
    return "\n".join([header] + body)


@dataclass
class NotebookTimeSeriesAgent:
    tactic: object

    def run(
        self,
        csv_path: str | Path,
        timestamp_col: str = "date",
        value_col: str = "sales",
        horizon: int = 7,
        frequency: str = "D",
        objective: str = "Detect anomalies and forecast future values.",
    ) -> dict:
        series_data = read_csv_as_text(Path(csv_path))
        task = TimeSeriesTask(
            series_data=series_data,
            timestamp_col=timestamp_col,
            value_col=value_col,
            horizon=horizon,
            frequency=frequency,
            objective=objective,
        )
        result = self.tactic(task)
        return result.model_dump()


agent = NotebookTimeSeriesAgent(tactic=tactic)
print("Agent initialized.")

Agent initialized.


In [5]:
# Demo run with the package's sample CSV.
demo_csv = repo_root / "Packages" / "time_series_analytics" / "demo_sales.csv"

output = agent.run(
    demo_csv,
    timestamp_col="date",
    value_col="sales",
    horizon=7,
    frequency="D",
    objective="Detect anomalies and forecast next-week sales demand.",
)

output

/var/folders/jd/2f22g_39093cpmq28d8lnw_w0000gq/T/ipykernel_70047/3331148048.py:46: UserWarning: No LogStore configured for tactic 'time_series_analysis'. Session data will not be persisted. Pass a LogStore instance via the log_store parameter.
  result = self.tactic(task)


{'summary': 'Sales demand over 2026-05-01 to 2026-05-10 shows a 10-day baseline of ~122 units with a sharp anomaly on May 5 (180 units, 48% above local mean). Post-anomaly data suggests a modest upward trend of ~0.79 units/day, lifting the baseline to 126–131 units. The forecast projects demand to climb gradually from 135 to 140 units over the next 7 days. However, critical limitations apply: only 10 historical observations were available for model training, and backtesting revealed severe prediction-interval undercoverage (16.7% vs. 90% target), indicating the model is overconfident. Point estimates are plausible if the recent drift persists, but interval estimates are unreliable for risk management.',
 'key_patterns': ['Baseline demand ~122 units (days 1–4 and post-anomaly stabilization)',
  'Sharp single-day spike on 2026-05-05 to 180 units (48% above local mean); likely external event or data anomaly',
  'Post-spike elevation: days 6–10 average 126–131 units, suggesting either a re

In [6]:
import pandas as pd
from pathlib import Path

def save_ts_output_to_csv(result: dict, out_dir: str | Path = ".", prefix: str = "ts_result"):
    '''
    Transform the output results to csv files. The csv files are saved in out_dir folder.
    '''
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    # 1) Forecast table
    forecast_df = pd.DataFrame(result.get("forecast", []))
    forecast_path = out_dir / f"{prefix}_forecast.csv"
    forecast_df.to_csv(forecast_path, index=False)

    # 2) Data quality issues
    dqi_df = pd.DataFrame(result.get("data_quality_issues", []))
    dqi_path = out_dir / f"{prefix}_data_quality_issues.csv"
    dqi_df.to_csv(dqi_path, index=False)

    # 3) Anomalies table
    anomalies_df = pd.DataFrame(result.get("anomalies", []))
    anomalies_path = out_dir / f"{prefix}_anomalies.csv"
    anomalies_df.to_csv(anomalies_path, index=False)

    # 4) One-row summary/meta table
    summary_row = {
        "summary": result.get("summary"),
        "confidence_note": result.get("confidence_note"),
        "key_patterns": " | ".join(result.get("key_patterns", [])),
        "recommendations": " | ".join(result.get("recommendations", [])),
    }
    summary_df = pd.DataFrame([summary_row])
    summary_path = out_dir / f"{prefix}_summary.csv"
    summary_df.to_csv(summary_path, index=False)

    return {
        "forecast": str(forecast_path),
        "data_quality_issues": str(dqi_path),
        "anomalies": str(anomalies_path),
        "summary": str(summary_path),
    }

# Example: save your output dict
paths = save_ts_output_to_csv(output, out_dir="Packages/time_series_analytics", prefix="demo_run")
paths

{'forecast': 'Packages/time_series_analytics/demo_run_forecast.csv',
 'data_quality_issues': 'Packages/time_series_analytics/demo_run_data_quality_issues.csv',
 'anomalies': 'Packages/time_series_analytics/demo_run_anomalies.csv',
 'summary': 'Packages/time_series_analytics/demo_run_summary.csv'}

In [7]:
single_df = pd.json_normalize(output, sep="_")
single_df.to_csv("Packages/time_series_analytics/demo_run_single.csv", index=False)

In [22]:
# Replace this with your own file path and columns.
my_csv = demo_csv  # e.g. Path("/absolute/path/to/your_series.csv")

my_output = agent.run(
    my_csv,
    timestamp_col="date",      # change to your timestamp column
    value_col="sales",         # change to your metric column
    horizon=14,
    frequency="D",
    objective="Create anomaly report and 14-step forecast for operations planning.",
)

my_output

{'summary': 'This 14-day operational forecast projects steady growth from 135 to 146 units (7.6% cumulative increase) based on a gentle upward trend observed in the post-anomaly period. However, forecast reliability is severely compromised by limited training data (10 observations), a major spike anomaly on May 5, and critically failed prediction intervals (16.7% coverage vs. 90% target). The central forecast is suitable for baseline planning, but uncertainty bounds are unreliable and should be manually widened by 3–5×. Immediate investigation of the May 5 spike is essential to determine if it represents a recurring event or one-time occurrence.',
 'key_patterns': ['Gentle upward trend of approximately +0.8 units/day observed in days 6–10 (post-anomaly stabilization at 127–131 range)',
  'Low baseline volatility (~3.1 units std dev excluding the spike) suggests relatively stable demand under normal conditions',
  'Mean reversion behavior after the May 5 spike, with sales returning to 1

## Validate forecast values against the statistical model

The pipeline computes the forecast and anomalies with a real statistical model
(`run_statistical_forecast`) and writes them back into the final report verbatim,
so the LLM never changes the numbers. The cells below run the full pipeline on
`demo_sales.csv` and confirm the reported forecast matches the statistical model
exactly, then run structural and anomaly sanity checks.

In [23]:
# Ground truth: the statistical model's forecast, computed directly (no LLM).
from tactics import run_statistical_forecast
import pandas as pd

demo_csv = repo_root / "Packages" / "time_series_analytics" / "demo_sales.csv"
series_text = read_csv_as_text(Path(demo_csv))
stat = run_statistical_forecast(
    series_text,
    timestamp_col="date",
    value_col="sales",
    horizon=7,
    frequency="D",
)
print("method:", stat.method)
print("diagnostics:")
print(stat.diagnostics_text())
stat_fc = pd.DataFrame(stat.points)
stat_fc

method: Holt's linear trend (additive trend, no seasonality)
diagnostics:
- n_observations: 10
- model: statsmodels.ExponentialSmoothing
- trend: additive
- seasonality: none (insufficient data for a full seasonal cycle)
- seasonal_periods: 1
- residual_std: 2.8125
- in_sample_mae: 9.8993
- confidence_level: 0.9
- non_negative_clamped: True
- n_anomalies: 1


,step,expected_value,lower_bound,upper_bound
0,1,135.2668,130.6406,139.8931
1,2,136.0608,129.5183,142.6033
2,3,136.8548,128.8419,144.8677
3,4,137.6488,128.3963,146.9013
4,5,138.4428,128.0982,148.7874
5,6,139.2367,127.9048,150.5687
6,7,140.0307,127.7908,152.2706


In [24]:
# Full pipeline: statistical forecast -> profile -> interpret -> synthesize.
# (Three sequential LLM calls; expect this to take a couple of minutes.)
pipeline_output = agent.run(
    demo_csv,
    timestamp_col="date",
    value_col="sales",
    horizon=7,
    frequency="D",
    objective="Detect anomalies and forecast next-week sales demand.",
)
pipeline_fc = pd.DataFrame(pipeline_output["forecast"])
pipeline_fc

,step,expected_value,lower_bound,upper_bound
0,1,135.2668,130.6406,139.8931
1,2,136.0608,129.5183,142.6033
2,3,136.8548,128.8419,144.8677
3,4,137.6488,128.3963,146.9013
4,5,138.4428,128.0982,148.7874
5,6,139.2367,127.9048,150.5687
6,7,140.0307,127.7908,152.2706


In [25]:
# The reported forecast must equal the statistical model's output exactly.
import numpy as np

cols = ["expected_value", "lower_bound", "upper_bound"]
merged = stat_fc.merge(pipeline_fc, on="step", suffixes=("_stat", "_pipeline"))
for c in cols:
    merged[c + "_match"] = np.isclose(
        merged[c + "_stat"], merged[c + "_pipeline"], atol=1e-6
    )

all_match = bool(merged[[c + "_match" for c in cols]].to_numpy().all())
print("Forecast values match the statistical model:", all_match)
merged

Forecast values match the statistical model: True


,step,expected_value_stat,lower_bound_stat,upper_bound_stat,expected_value_pipeline,lower_bound_pipeline,upper_bound_pipeline,expected_value_match,lower_bound_match,upper_bound_match
0,1,135.2668,130.6406,139.8931,135.2668,130.6406,139.8931,True,True,True
1,2,136.0608,129.5183,142.6033,136.0608,129.5183,142.6033,True,True,True
2,3,136.8548,128.8419,144.8677,136.8548,128.8419,144.8677,True,True,True
3,4,137.6488,128.3963,146.9013,137.6488,128.3963,146.9013,True,True,True
4,5,138.4428,128.0982,148.7874,138.4428,128.0982,148.7874,True,True,True
5,6,139.2367,127.9048,150.5687,139.2367,127.9048,150.5687,True,True,True
6,7,140.0307,127.7908,152.2706,140.0307,127.7908,152.2706,True,True,True


In [26]:
# Structural sanity checks + anomaly expectation.
fc = pipeline_output["forecast"]
assert len(fc) == 7, f"expected 7 forecast steps, got {len(fc)}"
for p in fc:
    assert p["lower_bound"] <= p["expected_value"] <= p["upper_bound"], (
        f"bounds inconsistent at step {p['step']}"
    )
    assert p["lower_bound"] >= 0, "sales forecast should be non-negative"

# The injected spike on 2026-05-05 should be flagged as an anomaly.
anom_dates = [a["start"] for a in pipeline_output["anomalies"]]
print("anomaly start dates:", anom_dates)
assert "2026-05-05" in anom_dates, "expected the 2026-05-05 spike to be flagged"
print("All checks passed.")

anomaly start dates: ['2026-05-05']
All checks passed.


## Backtesting & accuracy metrics

The pipeline now runs a rolling-origin backtest (expanding-window train, forecast,
then score against held-out actuals) and reports out-of-sample accuracy in
`backtest_metrics`: MAE, RMSE, sMAPE/MAPE, prediction-interval coverage, and mean
interval width. These numbers are computed in code and used by the LLM only to
calibrate the confidence note.

Note: the demo series has just 10 points, so only one short fold is possible and the
metrics are illustrative (the held-out window includes the injected spike, which
inflates the error and lowers coverage). Longer series produce multiple folds.

In [8]:
# Backtest metrics reported by the full pipeline (from the demo run above).
import pandas as pd

bt = output.get("backtest_metrics")
print("Reported backtest metrics:", bt)
pd.DataFrame([bt]) if bt else "(no backtest — insufficient history)"

Reported backtest metrics: {'mae': 14.8333, 'rmse': 24.4506, 'smape': 10.5157, 'mape': 9.3704, 'coverage': 0.1667, 'mean_interval_width': 8.8053, 'n_splits': 1, 'test_horizon': 6, 'n_test_points': 6}


,mae,rmse,smape,mape,coverage,mean_interval_width,n_splits,test_horizon,n_test_points
0,14.8333,24.4506,10.5157,9.3704,0.1667,8.8053,1,6,6


In [9]:
# Ground truth: run the statistical backtest directly and confirm the pipeline
# reports the same numbers, plus basic sanity checks.
from tactics import run_statistical_forecast

series_text = read_csv_as_text(Path(demo_csv))
stat = run_statistical_forecast(
    series_text, timestamp_col="date", value_col="sales", horizon=7, frequency="D"
)
gt = stat.backtest_metrics
print("Ground-truth backtest metrics:", gt)

if gt is None:
    print("No backtest possible for this series (too short).")
else:
    for k in ("mae", "rmse", "smape", "coverage", "n_splits"):
        assert output["backtest_metrics"][k] == gt[k], f"pipeline mismatch on {k}"
    assert gt["rmse"] >= gt["mae"] - 1e-9, "RMSE should be >= MAE"
    assert 0.0 <= gt["coverage"] <= 1.0, "coverage must be a fraction"
    assert gt["n_splits"] >= 1, "expected at least one backtest fold"
    print("All backtest checks passed.")

Ground-truth backtest metrics: {'mae': 14.8333, 'rmse': 24.4506, 'smape': 10.5157, 'mape': 9.3704, 'coverage': 0.1667, 'mean_interval_width': 8.8053, 'n_splits': 1, 'test_horizon': 6, 'n_test_points': 6}
All backtest checks passed.


## Frequency inference & gap handling

Before modeling, the package sorts the series, averages duplicate timestamps,
infers the frequency (`pandas.infer_freq`), reindexes to a complete grid, and
fills gaps using `fill_method` (`none`, `ffill`, `linear_interpolate`, `median`,
or `drop`). Every action is recorded in `data_quality_issues` so nothing is
silently distorted. The cells below feed a deliberately messy series (unsorted
rows, a duplicate timestamp, and a missing day) and show what gets recorded.

In [ ]:
# Statistical layer only (no LLM). Messy input: unsorted rows, a duplicate
# timestamp on 2026-05-02, and a missing day 2026-05-04.
from tactics import run_statistical_forecast
import pandas as pd

messy_csv = (
    "date,sales\n"
    "2026-05-03,122\n"
    "2026-05-01,120\n"
    "2026-05-02,125\n"
    "2026-05-02,127\n"
    "2026-05-05,180\n"
    "2026-05-06,124\n"
    "2026-05-07,126\n"
    "2026-05-08,129\n"
)
stat_msg = run_statistical_forecast(
    messy_csv, timestamp_col="date", value_col="sales", horizon=3,
    frequency="D", fill_method="linear_interpolate",
)
print("inferred:", stat_msg.inferred_frequency, "| effective:", stat_msg.effective_frequency)
print("n_duplicates:", stat_msg.diagnostics["n_duplicates"],
      "| n_gaps:", stat_msg.diagnostics["n_gaps"],
      "| n_filled:", stat_msg.diagnostics["n_filled"])
pd.DataFrame(stat_msg.data_quality_issues)

In [ ]:
# Sanity checks on the recorded preprocessing actions.
issues = {d["issue"] for d in stat_msg.data_quality_issues}
assert stat_msg.diagnostics["n_duplicates"] == 1, "expected one duplicate timestamp"
assert stat_msg.diagnostics["n_gaps"] == 1, "expected one missing day"
assert any("duplicate" in i for i in issues), "dedup should be recorded"
assert any("missing" in i for i in issues), "gap should be recorded"
assert any("out of order" in i for i in issues), "sort should be recorded"
print("Frequency/gap preprocessing recorded correctly.")

In [ ]:
# Full pipeline on the messy series (LLM; ~1-2 min). The preprocessing facts
# are prepended to data_quality_issues authoritatively.
from tactics.time_series_analysis import TimeSeriesTask

messy_task = TimeSeriesTask(
    series_data=messy_csv, timestamp_col="date", value_col="sales",
    horizon=3, frequency="D", fill_method="linear_interpolate",
)
messy_output = tactic(messy_task).model_dump()
print("data_quality_issues reported by the pipeline:")
pd.DataFrame(messy_output["data_quality_issues"])